# Flood Mapping with Sentinel-1 Polarimetric SAR - 2. Accuracy Assessment

Loads the change-detection outputs saved by `01_change_detection.ipynb`
(`<name>_<date>.npy` in the results folder), compares them against reference
flood / no-flood masks, and reports ROC / AUC per detector and land-cover class.

> Isundwa et al., *Analytical Optimization of Polarimetric Change Detectors for
> Mapping Open and Grassland Flooding Using Sentinel-1. Case of Rupununi, Guyana.*
> IEEE JSTARS, 2026.

## 1. Imports

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import auc

try:
    from tqdm import tqdm   # plain tqdm works in console and notebook
except ImportError:                    # no-op fallback if tqdm is missing
    class _NullBar:
        def __init__(self, iterable=None, **k):
            self.iterable = iterable
        def __iter__(self):
            return iter(self.iterable or [])
        def update(self, n=1):
            pass
        def set_postfix_str(self, s):
            pass
        def close(self):
            pass
    def tqdm(iterable=None, **k):
        return _NullBar(iterable, **k)

CSV_COLUMNS = ['Pd', 'Pf', 'threshold', 'accuracy', 'fscore', 'precision']

## 2. Load the detector outputs

Point `RESULTS_DIR` at the **same folder** you saved to in notebook 1, and set
`DATE` to the acquisition tag you used. `load_detectors` reads every
`<name>_<date>.npy` it finds (or pass a list of names for a subset).

In [ ]:
def load_detectors(results_dir, date, names=None):
    """Load detector arrays written by change_detection.save_detectors."""
    results_dir = str(results_dir)
    out = {}
    if names is None:
        suffix = f'_{date}.npy'
        for p in sorted(glob.glob(os.path.join(results_dir, f'*{suffix}'))):
            name = os.path.basename(p)[: -len(suffix)]
            out[name] = np.load(p)
    else:
        for name in names:
            out[name] = np.load(os.path.join(results_dir, f'{name}_{date}.npy'))
    print(f'Loaded {len(out)} detectors for {date} from {os.path.abspath(results_dir)}')
    return out


# --- Must match what you used in notebook 1 ---------------------------------
DATE = '20170715'
# Default assumes this notebook is run from the repo's notebooks/ folder, so
# '../results' is the repo results/. Use an absolute path for your own setup.
RESULTS_DIR = os.path.join('..', 'results')

det = load_detectors(RESULTS_DIR, DATE)
for k, v in det.items():
    print(f'  {k:8s} {v.shape}  {v.dtype}')

## 3. Load the reference masks

Four boolean masks digitised from the optical / drone reference data:

| key | file | meaning |
|---|---|---|
| `grass_flood`   | `Grassland.npy`          | flooded grassland |
| `open_flood`    | `Open_Flood.npy`         | open-water flood |
| `grass_noflood` | `Grassland_No_Flood.npy` | dry grassland |
| `open_noflood`  | `Open_No_Flood.npy`      | dry open areas |

Set `TRANSPOSE = True` if the saved detectors and the masks have opposite
(rows, cols) orientation — the original pipeline stored detectors as
`(samples, lines)` and masks as `(lines, samples)`, so a transpose is usually
needed. The shape check below will tell you if you have it right.

In [ ]:
MASK_DIR = os.path.join('..', 'data', 'accuracy_assessment_masks')   # <- edit if needed

masks = {
    'grass_flood':   np.load(os.path.join(MASK_DIR, 'Grassland_Flood.npy')).astype(bool),
    'open_flood':    np.load(os.path.join(MASK_DIR, 'Open_Flood.npy')).astype(bool),
    'grass_noflood': np.load(os.path.join(MASK_DIR, 'Grassland_No_Flood.npy')).astype(bool),
    'open_noflood':  np.load(os.path.join(MASK_DIR, 'Open_No_Flood.npy')).astype(bool),
}

TRANSPOSE = True   # apply .T to each detector so it matches the masks

# --- shape sanity check ---
a_det = next(iter(det.values()))
det_shape = a_det.T.shape if TRANSPOSE else a_det.shape
mask_shape = masks['grass_flood'].shape
print('detector shape (after TRANSPOSE):', det_shape)
print('mask shape                      :', mask_shape)
assert det_shape == mask_shape, (
    'Shape mismatch - toggle TRANSPOSE or check you loaded the right date/masks')
for k, m in masks.items():
    print(f'  {k:14s} {int(m.sum()):>10,} pixels')

## 4. Core accuracy function

For one detector / cover / direction, sweep `dim` thresholds between two
percentiles and record Pd, Pf, threshold, accuracy, F-score, precision.
`condition='Greater'` means flood = detector > t; `'Less_than'` the reverse.
(The threshold sweep uses `np.searchsorted` — identical counts to a per-threshold
mask loop, but fast on a full scene.)

In [ ]:
def accuracy_assessment(detector, flood_mask, no_flood_mask, dim, condition,
                        save_path=None, pct=(0.1, 93.0)):
    """Sweep dim thresholds; return a DataFrame of ROC metrics (optionally save CSV)."""
    detector = np.asarray(detector)
    flood_mask = np.asarray(flood_mask, dtype=bool)
    no_flood_mask = np.asarray(no_flood_mask, dtype=bool)

    sum_inf = float(flood_mask.sum())
    sum_clean = float(no_flood_mask.sum())

    fv = detector[flood_mask]
    nv = detector[no_flood_mask]
    fv = np.sort(fv[~np.isnan(fv)])
    nv = np.sort(nv[~np.isnan(nv)])

    start = np.nanpercentile(detector, pct[0])
    stop = np.nanpercentile(detector, pct[1])
    t = np.linspace(start, stop, num=dim)

    if condition == 'Greater':
        N_det = fv.size - np.searchsorted(fv, t, side='right')
        N_false = nv.size - np.searchsorted(nv, t, side='right')
    elif condition == 'Less_than':
        N_det = np.searchsorted(fv, t, side='left')
        N_false = np.searchsorted(nv, t, side='left')
    else:
        raise ValueError("condition must be 'Greater' or 'Less_than'")

    with np.errstate(divide='ignore', invalid='ignore'):
        Pd = N_det / sum_inf
        Pf = N_false / sum_clean
        t_neg = 1.0 - Pf
        f_neg = 1.0 - Pd
        prec = Pd / (Pd + Pf)
        rec = Pd / (Pd + f_neg)
        fscore = 2.0 * (prec * rec) / (prec + rec)
        acc = (Pd + t_neg) / 2.0

    df = pd.DataFrame(np.column_stack([Pd, Pf, t, acc, fscore, prec]),
                      columns=CSV_COLUMNS)
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        df.to_csv(save_path, mode='w', header=False, index=False)
    return df


def auc_from_df(df):
    """Trapezoidal AUC of the ROC implied by a metrics DataFrame."""
    order = np.argsort(df['Pf'].values)
    x = df['Pf'].values[order]
    y = df['Pd'].values[order]
    keep = ~(np.isnan(x) | np.isnan(y))
    x, y = x[keep], y[keep]
    if x.size < 2:
        return np.nan
    return auc(x, y)

## 5. Land-cover class masks

Each cover type combines the four base masks differently. Optionally, detected
open-flood pixels can be added to the grassland no-flood set (as in the paper,
to stop open water leaking into the grassland score).

In [ ]:
def cover_masks(masks, cover, open_detected_mask=None):
    gf = masks['grass_flood'];  of = masks['open_flood']
    gnf = masks['grass_noflood']; onf = masks['open_noflood']
    if cover == 'Combined':
        flood = gf | of
        no_flood = gnf | onf
    elif cover == 'Grassland':
        flood = gf
        no_flood = gnf | onf | of
        if open_detected_mask is not None:
            no_flood = no_flood | open_detected_mask.astype(bool)
    elif cover == 'open_flood':
        flood = of
        no_flood = gnf | onf | gf
    else:
        raise ValueError(f'unknown cover type: {cover}')
    return flood, no_flood


# Optional open-flood mask added to the grassland no-flood set.
# RatDif is negative for open flood, so e.g.:
#     OPEN_DETECTED = (det['RatDif'].T if TRANSPOSE else det['RatDif']) < -0.01
OPEN_DETECTED = None

## 6. Run the assessment for every detector

Loops all detectors x 3 cover classes x 2 directions and writes one CSV each to
`<OUT_DIR>/<condition>/<cover>_<condition>_<detector>_<date>.csv`.

In [ ]:
def assess_all_detectors(detectors, masks, out_path, date, dim=1000,
                         conditions=('Greater', 'Less_than'),
                         covers=('Grassland', 'open_flood', 'Combined'),
                         open_detected_mask=None, transpose=False):
    out_path = str(out_path)
    total = len(detectors) * len(covers) * len(conditions)
    pbar = tqdm(total=total, desc='accuracy')
    for name, d0 in detectors.items():
        d = np.asarray(d0).T if transpose else np.asarray(d0)
        for cover in covers:
            flood, no_flood = cover_masks(masks, cover, open_detected_mask)
            for condition in conditions:
                save_path = os.path.join(
                    out_path, condition, f'{cover}_{condition}_{name}_{date}.csv')
                accuracy_assessment(d, flood, no_flood, dim, condition, save_path)
                pbar.set_postfix_str(f'{name}/{cover}/{condition}')
                pbar.update(1)
    pbar.close()
    print(f'Wrote {len(detectors)} detectors x {len(covers)} covers x '
          f'{len(conditions)} conditions to {os.path.abspath(out_path)}')


OUT_DIR = os.path.join(RESULTS_DIR, 'accuracy_csv')
dim = 1000

assess_all_detectors(det, masks, OUT_DIR, DATE, dim=dim,
                     open_detected_mask=OPEN_DETECTED, transpose=TRANSPOSE)

## 7. AUC summary table

For each detector and cover class, the best threshold direction is chosen
automatically (the one giving the higher AUC), mirroring Table II of the paper.

In [ ]:
def auc_summary(detectors, masks, dim=1000,
                conditions=('Greater', 'Less_than'),
                covers=('Grassland', 'open_flood', 'Combined'),
                open_detected_mask=None, transpose=False):
    rows_auc, rows_dir = {}, {}
    for name, d0 in tqdm(detectors.items(), total=len(detectors), desc='AUC'):
        d = np.asarray(d0).T if transpose else np.asarray(d0)
        auc_row, dir_row = {}, {}
        for cover in covers:
            flood, no_flood = cover_masks(masks, cover, open_detected_mask)
            best_auc, best_c = -np.inf, None
            for condition in conditions:
                df = accuracy_assessment(d, flood, no_flood, dim, condition)
                a = auc_from_df(df)
                if np.isfinite(a) and a > best_auc:
                    best_auc, best_c = a, condition
            auc_row[cover] = best_auc if np.isfinite(best_auc) else np.nan
            dir_row[cover] = best_c
        rows_auc[name] = auc_row
        rows_dir[name] = dir_row
    table = pd.DataFrame(rows_auc).T[list(covers)]
    best_dir = pd.DataFrame(rows_dir).T[list(covers)]
    return table, best_dir


table, best_dir = auc_summary(det, masks, dim=dim,
                             open_detected_mask=OPEN_DETECTED, transpose=TRANSPOSE)
table.to_csv(os.path.join(OUT_DIR, f'AUC_summary_{DATE}.csv'))
print('Best threshold direction per detector:')
print(best_dir)
table.round(3)

## 8. ROC overview plot

Best-direction ROC of every detector for one cover class (log10 Pf vs Pd),
like Fig. 8 of the paper.

In [ ]:
def plot_roc_overview(detectors, masks, cover='Combined', dim=1000,
                      conditions=('Greater', 'Less_than'),
                      open_detected_mask=None, transpose=False,
                      save_path=None, title=None):
    flood, no_flood = cover_masks(masks, cover, open_detected_mask)
    plt.figure(figsize=(10, 7))
    for name, d0 in detectors.items():
        d = np.asarray(d0).T if transpose else np.asarray(d0)
        best_auc, best_df = -np.inf, None
        for condition in conditions:
            df = accuracy_assessment(d, flood, no_flood, dim, condition)
            a = auc_from_df(df)
            if np.isfinite(a) and a > best_auc:
                best_auc, best_df = a, df
        if best_df is None:
            continue
        order = np.argsort(best_df['Pf'].values)
        x = best_df['Pf'].values[order]
        y = best_df['Pd'].values[order]
        with np.errstate(divide='ignore'):
            plt.plot(np.log10(x), y, label=f'{name} (AUC={best_auc:.3f})')
    plt.title(title or f'ROC overview - {cover}', size=16)
    plt.xlabel('log10 False Alarm Rate (Pf)', size=13)
    plt.ylabel('Detection Rate (Pd)', size=13)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150)
    return plt.gcf()


for cover in ['Combined', 'open_flood', 'Grassland']:
    plot_roc_overview(det, masks, cover=cover, dim=dim,
                      open_detected_mask=OPEN_DETECTED, transpose=TRANSPOSE,
                      save_path=os.path.join(OUT_DIR, f'roc_{cover}_{DATE}.png'),
                      title=f'ROC overview - {cover}  ({DATE})')
    plt.show()

## 9. Best operating point per detector (threshold selection)

This reproduces the paper's two-stage selection:

1. **AUC screen** — drop any detector whose AUC <= `auc_gate` (default 0.6).
2. **Operating point** — among thresholds that clear the per-cover floors
   (PF / PD / ACC), keep the one with the highest `sort_by` metric (default
   accuracy). Its `THR` is the selected threshold ('best parameter').

Floors default to the values used in the original script (Grassland is looser);
edit `DEFAULT_FLOORS` or pass `floors=` to use the paper's stricter
PD>=0.85 / ACC>=0.85 / PF<=0.20 instead. `auc_method='roc'` uses proper ROC
ordering; pass `auc_method='sortboth'` to match the original script's AUC exactly.

In [ ]:
def _auc_value(df, method='roc'):
    """'roc' = PD ordered by PF (correct); 'sortboth' = original script's method."""
    if method == 'sortboth':
        x = np.sort(df['Pf'].values)
        y = np.sort(df['Pd'].values)
        keep = ~(np.isnan(x) | np.isnan(y))
        return auc(x[keep], y[keep]) if keep.sum() >= 2 else np.nan
    return auc_from_df(df)


DEFAULT_FLOORS = {
    'Grassland':  dict(pf=0.30, pd=0.60, acc=0.60),
    'open_flood': dict(pf=0.15, pd=0.70, acc=0.70),
    'Combined':   dict(pf=0.15, pd=0.70, acc=0.70),
}


def best_operating_points(detectors, masks, dim=1000,
                          conditions=('Greater', 'Less_than'),
                          covers=('Grassland', 'open_flood', 'Combined'),
                          open_detected_mask=None, transpose=False,
                          auc_gate=0.6, auc_method='roc',
                          floors=None, sort_by='accuracy'):
    """Screen by AUC, then pick the best-accuracy threshold per detector/cover/condition."""
    floors = floors or DEFAULT_FLOORS
    combos = [(n, c, cond) for n in detectors for c in covers for cond in conditions]
    rows = []
    for name, cover, condition in tqdm(combos, desc='best-params'):
        d = np.asarray(detectors[name]).T if transpose else np.asarray(detectors[name])
        flood, no_flood = cover_masks(masks, cover, open_detected_mask)
        df = accuracy_assessment(d, flood, no_flood, dim, condition)
        a = _auc_value(df, auc_method)
        if not np.isfinite(a) or a <= auc_gate:
            continue
        fl = floors[cover]
        sel = df[(df['Pf'] < fl['pf']) & (df['Pd'] > fl['pd']) & (df['accuracy'] > fl['acc'])]
        if sel.empty:                       # no threshold clears the floors -> skip
            continue
        best = sel.sort_values(sort_by, ascending=False).iloc[0]
        rows.append({'Detector': name, 'cover': cover, 'condition': condition,
                     'AUC': round(float(a), 4), 'THR': float(best['threshold']),
                     'PD': float(best['Pd']), 'PF': float(best['Pf']),
                     'ACC': float(best['accuracy']), 'F1': float(best['fscore']),
                     'PREC': float(best['precision'])})
    return pd.DataFrame(rows, columns=['Detector', 'cover', 'condition', 'AUC',
                                       'THR', 'PD', 'PF', 'ACC', 'F1', 'PREC'])


best_ops = best_operating_points(det, masks, dim=dim,
                                 open_detected_mask=OPEN_DETECTED, transpose=TRANSPOSE,
                                 auc_gate=0.6, auc_method='roc')

best_ops.to_csv(os.path.join(OUT_DIR, f'best_operating_points_{DATE}.csv'), index=False)
print(f'{len(best_ops)} detector/cover/condition combinations passed the screen')
best_ops